In [ ]:
import pathlib
import os
import datetime

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"

import torch
from torch.utils.tensorboard.writer import SummaryWriter
import trimesh

import network, utils, dataset, train

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [29]:
# Model name
current_time = datetime.datetime.now().strftime("%b%d_%H-%M")
# denoiser_name = "Resnet_2Blocks_256Hidden"
denoiser_name = "Resnet_2Blocks_256Hidden_Mixed_loss(0.5)_5epochs_1batch"
experiment_name = f"{current_time}_{denoiser_name}"


config = {
    'experiment_name': experiment_name,
    'device': 'cuda:0',
    'is_overfit': True,
    'batch_size': 1,
    'resume_ckpt': None,
    'learning_rate': 0.0005,
    'lambda': 0.5,
    'max_epochs': 5,
    'timesteps': 1000,
    'print_every_n': 1, # every n batches
    'validate_every_n': 25,
    'print_EMD_every_n': 1
}

In [30]:
# declare device
device = torch.device('cpu')
if torch.cuda.is_available() and config['device'].startswith('cuda'):
    device = torch.device(config['device'])
    print('Using device:', config['device'])
else:
    print('Using CPU')

# create dataloaders
trainset = dataset.Dataset('train' if not config['is_overfit'] else 'overfit', config['timesteps'])
trainloader = torch.utils.data.DataLoader(trainset, batch_size=config['batch_size'], shuffle=True, num_workers=1)

# valset = dataset.Dataset('val' if not config['is_overfit'] else 'overfit', config['timesteps'])
# valloader = torch.utils.data.DataLoader(valset, batch_size=config['batch_size'], shuffle=False, num_workers=1)

denoiser = network.Denoiser()
diffuser = network.Diffuser(config['timesteps'])

# load model if resuming from checkpoint
if config['resume_ckpt'] is not None:
        utils.reload_model(denoiser, diffuser, config['experiment_name'], device)

# move model to specified device
denoiser.to(device)
diffuser.to(device)
optimizer = torch.optim.Adam(denoiser.parameters(), lr=config['learning_rate'])

# Create tensorboard writer    
log_path = pathlib.Path(f"logs/diffusion_training/{datetime.datetime.now().strftime('%b%d')}/{config['experiment_name']}")
writer = SummaryWriter(log_path)

#Run this code in terminal to start tensorboard: tensorboard --logdir=diffusion/nikola/logs/diffusion_training


total, trainable = utils.count_parameters(denoiser)
print(f"Total: {total:,} | Trainable: {trainable:,} | Model size: {utils.model_memory_size(denoiser):.3f} MB")

Using device: cuda:0
Total: 824,067 | Trainable: 824,067 | Model size: 3.144 MB


In [31]:
# start training
#tensorboard --logdir=diffusion/nikola/logs/diffusion_training

train.train(denoiser=denoiser, diffuser=diffuser, trainloader=trainloader, device=device, optimizer=optimizer, config=config, writer=writer,valloader=None)

[00/000] train_loss: 89.028
[00/001] train_loss: 26.826
[00/002] train_loss: 21.110
[00/003] train_loss: 23.515
[00/004] train_loss: 0.661
[00/005] train_loss: 0.538
[00/006] train_loss: 0.884
[00/007] train_loss: 0.471
[00/008] train_loss: 0.277
[00/009] train_loss: 0.440
[00/010] train_loss: 21.972
[00/011] train_loss: 43.369
[00/012] train_loss: 43.489
[00/013] train_loss: 1.062
[00/014] train_loss: 370.033
[00/015] train_loss: 0.551
[00/016] train_loss: 0.733
[00/017] train_loss: 1.782
[00/018] train_loss: 4.236
[00/019] train_loss: 1.710
[00/020] train_loss: 79.109
[00/021] train_loss: 26.005
[00/022] train_loss: 3.767
[00/023] train_loss: 24.964
[00/024] train_loss: 520.216
[00/025] train_loss: 534.427
[00/026] train_loss: 4.426
[00/027] train_loss: 6.691
[00/028] train_loss: 1.555
[00/029] train_loss: 32.116
[00/030] train_loss: 29.940
[00/031] train_loss: 0.961
[00/032] train_loss: 1.117
[00/033] train_loss: 1.399
[00/034] train_loss: 0.698
[00/035] train_loss: 5.717
[00/036] t

In [ ]:
#Load a model:
denoiser_id = "May13_14-53_Resnet_2Blocks_256Hidden_reweighted_loss_10000_epochs"
utils.reload_model(denoiser, diffuser, denoiser_id, device)

In [32]:
generateDDIM = True
generateDDPM = True
number_of_points = 2056
DDIM_steps = 40
number_of_DDIM_iterations = 1

if generateDDIM:
    for _ in range(number_of_DDIM_iterations):
        generated_pc_ddim = network.sample_ddim(denoiser, diffuser, n_points=number_of_points, steps=DDIM_steps)
        generated_pcd_ddim = trimesh.PointCloud(generated_pc_ddim.squeeze().cpu().numpy())
        utils.visualize_comparison(trainset[0], generated_pc_ddim, window_name="DDIM Target (Red) vs Generated (Blue)")

if generateDDPM:
    generated_pc_ddpm = network.sample_ddpm(denoiser, diffuser, n_points=number_of_points)
    generated_pcd_ddpm = trimesh.PointCloud(generated_pc_ddpm.squeeze().cpu().numpy())
    utils.visualize_comparison(trainset[0], generated_pc_ddpm, window_name="DDPM Target (Red) vs Generated (Blue)")

Visualizing: Target is RED, Generated is BLUE.
Visualizing: Target is RED, Generated is BLUE.


In [ ]:
generated_pc, samples_list = network.sample_and_capture(denoiser, diffuser, n_points=number_of_points, save_every=10)
utils.visualize_diffusion_progress(samples_list, window_name="Diffusion Process")

In [ ]:
# Optionally, save the generated point clouds to disk
generated_pcd_ddim.export(f"output/{denoiser_id}_ddim.obj")
generated_pcd_ddpm.export(f"output/{denoiser_id}_ddpm.obj")